In [1]:
#| default_exp core

# Core asterism search module
> Coordinate transforms, scoring functions, grid utilities, and GPU pipeline

In [2]:
#| export
import os

# CRITICAL: Set LD_LIBRARY_PATH BEFORE importing torch (for HIP kernel)
torch_lib_path = os.path.expanduser("~") + "/dev/amateur_astro/py-asterisms/.venv/lib/python3.12/site-packages/torch/lib"
if "LD_LIBRARY_PATH" in os.environ:
    if torch_lib_path not in os.environ["LD_LIBRARY_PATH"]:
        os.environ["LD_LIBRARY_PATH"] = f"{torch_lib_path}:{os.environ['LD_LIBRARY_PATH']}"
else:
    os.environ["LD_LIBRARY_PATH"] = torch_lib_path

# Fix GPU memory fragmentation for ROCm/HIP
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
import polars as pl
import numpy as np
import torch
import math
import time
import heapq
from typing import List, Any
import tqdm
import pickle
import pyarrow as pa
from dataclasses import dataclass, field

## Search configuration

| Mode | Max Mag | Max Extent | Grid Step | Search Radius |
|------|---------|------------|-----------|---------------|
| Naked eye | 6 | 20deg | 10deg | 20deg |
| Binocular | 11 | 6deg | 4deg | 6deg |
| Telescopic | 15 | 2deg | 4deg | 2deg |

In [ ]:
#| export
@dataclass
class SearchConfig:
    name: str
    max_mag: float
    max_extent_deg: float
    grid_step_deg: int
    search_radius_deg: int
    max_stars_per_region: int = 0  # 0 = no cap

NAKED_EYE = SearchConfig("naked_eye", 6, 20, 10, 20)
BINOCULAR = SearchConfig("binocular", 11, 6, 4, 6, max_stars_per_region=300)
TELESCOPIC = SearchConfig("telescopic", 15, 2, 4, 2)

## Data classes and coordinate transforms

In [ ]:
#| export
@dataclass(order=True)
class ScoreItem:
    score: float
    region: int=field(compare=False)
    item: Any=field(compare=False)
    
@dataclass(order=True)
class Points:
    points: Any=field(compare=False)
    

def convert_to_cartesian(distances, ra, dec):
    x = distances * torch.cos(dec) * torch.cos(ra)
    y = distances * torch.cos(dec) * torch.sin(ra)
    z = distances * torch.sin(dec)
    return torch.stack((x, y, z), dim=1)

def calculate_distances(coords):
    dist_matrix = torch.cdist(coords, coords)
    unique_distances = torch.unique(dist_matrix)
    return unique_distances[unique_distances > 0]

def distance_from_magnitude(m, M):
    return 10**((m - M + 5) / 5)

def distance_from_magnitude_tensor(m: torch.Tensor, M: torch.Tensor) -> torch.Tensor:
    return 10**((m - M + 5) / 5)

def tetrahedron_score(coords):
    distances = calculate_distances(coords)
    print(distances)
    mean_distance = torch.mean(distances)
    std_dev = torch.std(distances)
    score = math.exp(-std_dev.item() / mean_distance.item())
    return score

def square_score(tensor):
    """Return a measure of how close the points in a tensor are to forming a square."""
    spatial_distances = torch.pdist(tensor, p=2)
    print(spatial_distances)
    sorted_distances = torch.sort(spatial_distances)[0]
    std_of_smallest_4 = sorted_distances[:4].std().item()
    return std_of_smallest_4

def measure_squareness_old(tensor):
    spatial_distances = torch.pdist(tensor, p=2)
    distances = torch.sort(spatial_distances)[0]
    mean_sides = torch.mean(distances[:4])
    mean_diagonal = torch.mean(distances[4:])
    squareness = mean_diagonal / mean_sides
    return abs(1 - (squareness / torch.sqrt(torch.tensor(2.0)).item()))

def measure_squareness(tensor):
    spatial_distances = torch.pdist(tensor, p=2)
    distances = torch.sort(spatial_distances)[0]
    std_sides = torch.std(distances[:4])
    std_diagonal = torch.std(distances[4:])
    side_uniformity = torch.exp(-std_sides)
    diagonal_uniformity = torch.exp(-std_diagonal)
    mean_sides = torch.mean(distances[:4])
    mean_diagonal = torch.mean(distances[4:])
    squareness_ratio = mean_sides / mean_diagonal
    final_squareness = (squareness_ratio / torch.sqrt(torch.tensor(2.0)).item()) * side_uniformity * diagonal_uniformity
    return abs(1 - final_squareness.item())

def radecmag_to_cartesian(points_tensor):
    """Convert (RA_deg, Dec_deg, Vmag) to 3D using perceptual distance from magnitude.
    
    Uses distance modulus with M=0: d = 10^((m+5)/5).
    Normalized so the brightest star (lowest mag) in the group sits at r=1;
    only relative magnitude differences affect the 3D shape."""
    ra_rad = torch.deg2rad(points_tensor[:, 0])
    dec_rad = torch.deg2rad(points_tensor[:, 1])
    mag = points_tensor[:, 2]
    min_mag = mag.min()
    r = 10.0 ** ((mag - min_mag) / 5.0)
    x = r * torch.cos(dec_rad) * torch.cos(ra_rad)
    y = r * torch.cos(dec_rad) * torch.sin(ra_rad)
    z = r * torch.sin(dec_rad)
    return torch.stack([x, y, z], dim=1)

def radecmag_to_angular(points_tensor):
    """Convert (RA_deg, Dec_deg, Vmag) to unit vectors on celestial sphere.
    
    Chord distance between unit vectors is monotonic with angular separation,
    so equilateral in chord distance = equilateral on the sky."""
    ra_rad = torch.deg2rad(points_tensor[:, 0])
    dec_rad = torch.deg2rad(points_tensor[:, 1])
    x = torch.cos(dec_rad) * torch.cos(ra_rad)
    y = torch.cos(dec_rad) * torch.sin(ra_rad)
    z = torch.sin(dec_rad)
    return torch.stack([x, y, z], dim=1)

## Scoring functions

In [ ]:
#| export
def mass_score_triangle_torch(points_tensor, device='cpu', mode='3d'):
    points_tensor = points_tensor.to(device)

    if mode == '3d':
        scoring_coords = radecmag_to_cartesian(points_tensor)
    else:
        scoring_coords = radecmag_to_angular(points_tensor)

    idx_combinations = torch.combinations(torch.arange(scoring_coords.shape[0]), r=3)

    p1 = scoring_coords[idx_combinations[:, 0]]
    p2 = scoring_coords[idx_combinations[:, 1]]
    p3 = scoring_coords[idx_combinations[:, 2]]

    a = torch.linalg.norm(p2 - p1, dim=1)
    b = torch.linalg.norm(p3 - p2, dim=1)
    c = torch.linalg.norm(p1 - p3, dim=1)

    # Raw coefficient of variation (scale-invariant)
    mean = (a + b + c) / 3
    std_dev = torch.sqrt(((a - mean)**2 + (b - mean)**2 + (c - mean)**2) / 3)
    scores = torch.where(mean > 0, std_dev / mean, torch.tensor([1.], device=mean.device))

    # Return original RA/Dec/Mag points for storage
    p1_full = points_tensor[idx_combinations[:, 0]]
    p2_full = points_tensor[idx_combinations[:, 1]]
    p3_full = points_tensor[idx_combinations[:, 2]]
    points_combined = torch.stack([p1_full, p2_full, p3_full], dim=1)
    return scores, points_combined


# The 3 distinct quadrilateral cycles for vertices {0,1,2,3}.
# Each cycle defines 4 side-pairs and 2 diagonal-pairs.
_QUAD_CYCLES = [
    # cycle 0-1-2-3: sides (01,12,23,30), diags (02,13)
    ([0,1], [1,2], [2,3], [3,0], [0,2], [1,3]),
    # cycle 0-1-3-2: sides (01,13,32,20), diags (03,12)
    ([0,1], [1,3], [3,2], [2,0], [0,3], [1,2]),
    # cycle 0-2-1-3: sides (02,21,13,30), diags (01,23)
    ([0,2], [2,1], [1,3], [3,0], [0,1], [2,3]),
]


def mass_score_square_torch(points_tensor, device='cpu', mode='3d'):
    """Score all 4-star combinations for squareness, vectorized on GPU.
    
    Tests all 3 possible quadrilateral vertex orderings per combo and
    picks the one that scores best as a square (proper closed shape)."""
    points_tensor = points_tensor.to(device)

    if mode == '3d':
        scoring_coords = radecmag_to_cartesian(points_tensor)
    else:
        scoring_coords = radecmag_to_angular(points_tensor)

    idx_combinations = torch.combinations(torch.arange(scoring_coords.shape[0], device=device), r=4)

    # Get all 4-point groups: (N_combos, 4, 3)
    combos = scoring_coords[idx_combinations]
    N = combos.shape[0]
    sqrt2 = math.sqrt(2.0)

    best_scores = torch.full((N,), float('inf'), device=device)

    for cycle in _QUAD_CYCLES:
        s0, s1, s2, s3, d0, d1 = cycle
        side_dists = torch.stack([
            torch.linalg.norm(combos[:, s0[0]] - combos[:, s0[1]], dim=1),
            torch.linalg.norm(combos[:, s1[0]] - combos[:, s1[1]], dim=1),
            torch.linalg.norm(combos[:, s2[0]] - combos[:, s2[1]], dim=1),
            torch.linalg.norm(combos[:, s3[0]] - combos[:, s3[1]], dim=1),
        ], dim=1)  # (N, 4)
        diag_dists = torch.stack([
            torch.linalg.norm(combos[:, d0[0]] - combos[:, d0[1]], dim=1),
            torch.linalg.norm(combos[:, d1[0]] - combos[:, d1[1]], dim=1),
        ], dim=1)  # (N, 2)

        mean_sides = side_dists.mean(dim=1)
        mean_diags = diag_dists.mean(dim=1)
        cv_sides = side_dists.std(dim=1) / mean_sides.clamp(min=1e-12)
        cv_diags = diag_dists.std(dim=1) / mean_diags.clamp(min=1e-12)
        ratio_dev = torch.abs(mean_diags / mean_sides.clamp(min=1e-12) - sqrt2)
        cycle_scores = cv_sides + cv_diags + ratio_dev

        best_scores = torch.minimum(best_scores, cycle_scores)

    # Return original RA/Dec/Mag points for storage
    points_combined = points_tensor[idx_combinations]  # (N_combos, 4, 3)
    return best_scores, points_combined


def transform_radecmag_from_numpy(stars):
    torch_tensors = [torch.from_numpy(star) for star in stars]
    zeroes = torch.zeros(len(torch_tensors[2]))
    print("one ", torch_tensors)
    torch_tensors[2] = distance_from_magnitude_tensor(torch_tensors[2], zeroes)
    print("two", torch_tensors)
    coords = convert_to_cartesian(*torch_tensors)
    return coords

def global_normalize_tensor(tensor):
    """Normalize a tensor based on its global min and max values."""
    global_min = torch.min(tensor)
    global_max = torch.max(tensor)
    normalized = (tensor - global_min) / (global_max - global_min)
    return normalized
    
def radec_normalize_tensor(tensors):
    """Normalize tensors based on their global min and max values, excluding the 3rd column."""
    tensor = tensors[:, :2]
    global_min = torch.min(tensor)
    global_max = torch.max(tensor)
    normalized = (tensor - global_min) / (global_max - global_min)
    return normalized

def mag_score(tensor):
    max = tensor[:, 2].max()
    min = tensor[:, 2].min()
    return max - min

def score_triangle(tensor):    
    spatial_distances = torch.pdist(tensor, p=2)
    return torch.std(spatial_distances)

## Angular extent filtering

In [ ]:
#| export
def triangle_extent_deg(points_tensor):
    """Max pairwise angular separation in degrees for a triangle's 3 vertices.
    
    Args:
        points_tensor: shape (3, 3) tensor of (RA_deg, Dec_deg, Vmag)
    Returns:
        Maximum angular separation in degrees between any two vertices.
    """
    ra = torch.deg2rad(points_tensor[:, 0])
    dec = torch.deg2rad(points_tensor[:, 1])
    
    # Pairwise angular separation using spherical law of cosines
    max_sep = 0.0
    for i in range(3):
        for j in range(i+1, 3):
            cos_sep = (torch.sin(dec[i]) * torch.sin(dec[j]) +
                       torch.cos(dec[i]) * torch.cos(dec[j]) * torch.cos(ra[i] - ra[j]))
            cos_sep = torch.clamp(cos_sep, -1.0, 1.0)
            sep = torch.acos(cos_sep)
            max_sep = max(max_sep, torch.rad2deg(sep).item())
    return max_sep


def triangle_extent_deg_batch(points_batch):
    """Vectorized max pairwise angular separation for a batch of triangles.
    
    Args:
        points_batch: shape (N, 3, 3) tensor of (RA_deg, Dec_deg, Vmag)
    Returns:
        Tensor of shape (N,) with max angular separation in degrees per triangle.
    """
    ra = torch.deg2rad(points_batch[:, :, 0])   # (N, 3)
    dec = torch.deg2rad(points_batch[:, :, 1])   # (N, 3)
    
    # Pairs: (0,1), (0,2), (1,2)
    pairs = [(0, 1), (0, 2), (1, 2)]
    seps = []
    for i, j in pairs:
        cos_sep = (torch.sin(dec[:, i]) * torch.sin(dec[:, j]) +
                   torch.cos(dec[:, i]) * torch.cos(dec[:, j]) * torch.cos(ra[:, i] - ra[:, j]))
        cos_sep = torch.clamp(cos_sep, -1.0, 1.0)
        seps.append(torch.acos(cos_sep))
    
    all_seps = torch.stack(seps, dim=1)  # (N, 3)
    return torch.rad2deg(all_seps.max(dim=1).values)


def shape_extent_deg_batch(points_batch):
    """Vectorized max pairwise angular separation for a batch of any shape.
    
    Args:
        points_batch: shape (N, K, 3) tensor where K=3 (triangle) or K=4 (square)
    Returns:
        Tensor of shape (N,) with max angular separation in degrees.
    """
    K = points_batch.shape[1]
    ra = torch.deg2rad(points_batch[:, :, 0])   # (N, K)
    dec = torch.deg2rad(points_batch[:, :, 1])   # (N, K)
    
    seps = []
    for i in range(K):
        for j in range(i + 1, K):
            cos_sep = (torch.sin(dec[:, i]) * torch.sin(dec[:, j]) +
                       torch.cos(dec[:, i]) * torch.cos(dec[:, j]) * torch.cos(ra[:, i] - ra[:, j]))
            cos_sep = torch.clamp(cos_sep, -1.0, 1.0)
            seps.append(torch.acos(cos_sep))
    
    all_seps = torch.stack(seps, dim=1)
    return torch.rad2deg(all_seps.max(dim=1).values)



def compute_tilt_batch(points_batch):
    """Tilt angle of a shape relative to line of sight: 0° = face-on, 90° = edge-on.

    Args:
        points_batch: (N, K, 3) tensor of (RA_deg, Dec_deg, Vmag), K=3 or K=4
    Returns:
        (N,) tensor of tilt angles in degrees.
    """
    N, K, _ = points_batch.shape
    device = points_batch.device

    flat = points_batch.reshape(N * K, 3)
    ra_rad = torch.deg2rad(flat[:, 0])
    dec_rad = torch.deg2rad(flat[:, 1])
    mag = flat[:, 2].reshape(N, K)
    min_mag = mag.min(dim=1, keepdim=True).values
    r = 10.0 ** ((mag - min_mag) / 5.0)
    r_flat = r.reshape(N * K)

    x = r_flat * torch.cos(dec_rad) * torch.cos(ra_rad)
    y = r_flat * torch.cos(dec_rad) * torch.sin(ra_rad)
    z = r_flat * torch.sin(dec_rad)
    cart = torch.stack([x, y, z], dim=1).reshape(N, K, 3)

    if K == 3:
        normal = torch.cross(cart[:, 1] - cart[:, 0], cart[:, 2] - cart[:, 0], dim=1)
    elif K == 4:
        normal = torch.cross(cart[:, 2] - cart[:, 0], cart[:, 3] - cart[:, 1], dim=1)
    else:
        return torch.zeros(N, device=device)

    view = cart.mean(dim=1)

    normal_hat = normal / (torch.linalg.norm(normal, dim=1, keepdim=True) + 1e-12)
    view_hat = view / (torch.linalg.norm(view, dim=1, keepdim=True) + 1e-12)

    cos_angle = torch.abs((normal_hat * view_hat).sum(dim=1)).clamp(0.0, 1.0)
    tilt = torch.rad2deg(torch.acos(cos_angle))
    return tilt


def chain_extent_deg(points_tensor):
    """Max pairwise angular separation in degrees for a chain of K stars.
    
    Args:
        points_tensor: shape (K, 3) tensor of (RA_deg, Dec_deg, Vmag), K >= 4
    Returns:
        Maximum angular separation in degrees between any two stars.
    """
    K = len(points_tensor)
    ra = torch.deg2rad(points_tensor[:, 0])
    dec = torch.deg2rad(points_tensor[:, 1])
    max_sep = 0.0
    for i in range(K):
        for j in range(i + 1, K):
            cos_sep = (torch.sin(dec[i]) * torch.sin(dec[j]) +
                       torch.cos(dec[i]) * torch.cos(dec[j]) * torch.cos(ra[i] - ra[j]))
            cos_sep = torch.clamp(cos_sep, -1.0, 1.0)
            sep = torch.acos(cos_sep)
            max_sep = max(max_sep, torch.rad2deg(sep).item())
    return max_sep

## Collinear chain detection

Finds chains of 4+ stars lying on a great circle. Uses angle binning (O(n² log n))
instead of brute-force C(n,4) (O(n⁴)).

In [ ]:
#| export
from collections import defaultdict


def _to_unit_vectors(stars_tensor):
    """Convert (RA_deg, Dec_deg, Vmag) to unit vectors on the celestial sphere."""
    ra_rad = torch.deg2rad(stars_tensor[:, 0])
    dec_rad = torch.deg2rad(stars_tensor[:, 1])
    x = torch.cos(dec_rad) * torch.cos(ra_rad)
    y = torch.cos(dec_rad) * torch.sin(ra_rad)
    z = torch.sin(dec_rad)
    return torch.stack([x, y, z], dim=1)


def _to_unit_vectors_batch(stars_batch):
    """Convert (B, N, 3) batch of (RA_deg, Dec_deg, Vmag) to unit vectors."""
    ra_rad = torch.deg2rad(stars_batch[:, :, 0])
    dec_rad = torch.deg2rad(stars_batch[:, :, 1])
    x = torch.cos(dec_rad) * torch.cos(ra_rad)
    y = torch.cos(dec_rad) * torch.sin(ra_rad)
    z = torch.sin(dec_rad)
    return torch.stack([x, y, z], dim=2)


def _perpendicular_distance(unit_vecs, i_start, i_end):
    """RMS perpendicular distance of interior points from the great circle through endpoints.
    
    Returns (rms_perp_deg, max_perp_deg) for all points between start and end.
    """
    normal = torch.cross(unit_vecs[i_start], unit_vecs[i_end], dim=0)
    norm = torch.linalg.norm(normal)
    if norm < 1e-12:
        return 0.0, 0.0
    normal = normal / norm
    interior = [i for i in range(len(unit_vecs)) if i != i_start and i != i_end]
    if len(interior) == 0:
        return 0.0, 0.0
    dots = torch.abs(unit_vecs[interior] @ normal)
    dots = torch.clamp(dots, 0.0, 1.0)
    perp_rad = torch.asin(dots)
    perp_deg = torch.rad2deg(perp_rad)
    return perp_deg.pow(2).mean().sqrt().item(), perp_deg.max().item()


def _batch_compute_angles(padded_stars, counts, device):
    """Compute direction angle matrices for all regions in one GPU pass.
    
    Args:
        padded_stars: (B, max_N, 3) tensor of stars, zero-padded
        counts: (B,) int tensor of actual star counts per region
        device: GPU device
    Returns:
        sorted_angles: (B, max_N, max_N) sorted angle values (CPU numpy)
        sorted_indices: (B, max_N, max_N) sorted star indices (CPU numpy)
        unit_vecs: (B, max_N, 3) unit vectors (stays on GPU)
    """
    B, max_N, _ = padded_stars.shape

    unit_vecs = _to_unit_vectors_batch(padded_stars)  # (B, N, 3)

    ra_rad = torch.deg2rad(padded_stars[:, :, 0])   # (B, N)
    dec_rad = torch.deg2rad(padded_stars[:, :, 1])   # (B, N)

    # Tangent frames: (B, N, 3)
    east = torch.stack([
        -torch.sin(ra_rad), torch.cos(ra_rad), torch.zeros_like(ra_rad)
    ], dim=2)
    north = torch.stack([
        -torch.sin(dec_rad) * torch.cos(ra_rad),
        -torch.sin(dec_rad) * torch.sin(ra_rad),
        torch.cos(dec_rad),
    ], dim=2)

    # Diffs: (B, N, N, 3) = unit_vecs[b,j] - unit_vecs[b,i]
    diffs = unit_vecs.unsqueeze(1) - unit_vecs.unsqueeze(2)  # (B, N, N, 3)

    # Project out radial component: radial_dot = dot(diffs[b,i,j], unit_vecs[b,i])
    radial_dot = (diffs * unit_vecs.unsqueeze(2)).sum(dim=3, keepdim=True)  # (B, N, N, 1)
    tangent = diffs - radial_dot * unit_vecs.unsqueeze(2)  # (B, N, N, 3)

    # Project onto tangent frames
    tx = (tangent * east.unsqueeze(2)).sum(dim=3)   # (B, N, N)
    ty = (tangent * north.unsqueeze(2)).sum(dim=3)  # (B, N, N)
    angles = torch.atan2(ty, tx) % math.pi          # (B, N, N)

    # Mask out padding and self-diagonal: pi sorts to end of [0, pi) real values
    for b in range(B):
        n = counts[b]
        if n < max_N:
            angles[b, n:, :] = math.pi
            angles[b, :, n:] = math.pi
    diag_idx = torch.arange(max_N, device=device)
    angles[:, diag_idx, diag_idx] = math.pi

    sorted_angles, sorted_indices = torch.sort(angles, dim=2)

    return sorted_angles.cpu().numpy(), sorted_indices.cpu().numpy(), unit_vecs


def _find_chains_batch(sa_cpu, si_cpu, counts_cpu, angle_tol_rad):
    """Find collinear chain candidates across all regions.
    
    Uses numpy vectorized diff/cumsum for cluster detection instead of
    Python sliding windows. Only iterates in Python over rows that actually
    contain chains (pre-filtered via vectorized check).
    
    Returns list of (region_b, chain_indices_tuple).
    """
    B, max_N, _ = sa_cpu.shape
    all_candidates = []

    for b in range(B):
        N = int(counts_cpu[b])
        if N < 4:
            continue

        # After diagonal masking, positions 0..N-2 are other stars, N-1 is self
        M = N - 1
        if M < 3:
            continue

        # Extract this region's sorted angles and indices for all reference stars
        block_a = sa_cpu[b, :N, :M]  # (N, M) - other stars only
        block_i = si_cpu[b, :N, :M]  # (N, M)

        # Vectorized cluster detection: diff + cumsum for all rows at once
        diffs = np.diff(block_a, axis=1)  # (N, M-1)
        breaks = diffs > angle_tol_rad    # (N, M-1)

        # Pre-filter: only process rows with runs of 3+ (2+ consecutive non-breaks)
        if M >= 3:
            has_pair = ~breaks[:, :-1] & ~breaks[:, 1:]  # (N, M-2)
            row_has_chain = np.any(has_pair, axis=1)      # (N,)
        else:
            continue

        # Also check wrap-around: angles near 0 and near pi are close
        wrap_gap = block_a[:, 0] + (math.pi - block_a[:, M - 1])
        row_has_wrap = wrap_gap < angle_tol_rad

        active_rows = np.where(row_has_chain | row_has_wrap)[0]
        if len(active_rows) == 0:
            continue

        seen = set()

        for i in active_rows:
            row_a = block_a[i]   # (M,)
            row_idx = block_i[i] # (M,)

            # Cluster IDs via cumsum on breaks
            cluster_ids = np.empty(M, dtype=np.int32)
            cluster_ids[0] = 0
            cluster_ids[1:] = np.cumsum(breaks[i])

            # Handle wrap-around: merge last cluster into first
            if row_has_wrap[i] and cluster_ids[-1] > 0:
                last_cid = cluster_ids[-1]
                cluster_ids[cluster_ids == last_cid] = 0

            # Count cluster sizes via bincount
            max_cid = cluster_ids[-1] + 1
            sizes = np.bincount(cluster_ids, minlength=max_cid)

            # Process clusters with >= 3 members (+ reference star i = chain of 4+)
            large = np.where(sizes >= 3)[0]
            for cid in large:
                members = row_idx[cluster_ids == cid]
                chain = tuple(sorted(np.append(members, i)))
                if chain not in seen:
                    seen.add(chain)
                    all_candidates.append((b, chain))

    return all_candidates



def _find_smooth_chains(uv_np, max_turn_deg=30.0, min_stars=4, k_neighbors=15):
    """Find smooth chains (lines AND curves) using GPU-batched greedy path extension.
    
    Extends ALL starting pairs simultaneously using tensor operations.
    
    Args:
        uv_np: (N, 3) numpy array of unit vectors on celestial sphere
        max_turn_deg: maximum turning angle between consecutive segments
        min_stars: minimum chain length
        k_neighbors: number of nearest neighbors to try as starting direction
    
    Returns:
        List of chains, each a tuple of star indices (sorted)
    """
    N = len(uv_np)
    if N < min_stars:
        return []
    
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    uv = torch.tensor(uv_np, dtype=torch.float32, device=device)
    
    # Pairwise chord distances
    dots = uv @ uv.T
    dists = torch.sqrt(torch.clamp(2 - 2 * dots, min=0))
    dists.fill_diagonal_(float('inf'))
    
    k = min(k_neighbors, N - 1)
    knn = torch.argsort(dists, dim=1)[:, :k]  # (N, k)
    
    cos_max_turn = math.cos(math.radians(max_turn_deg))
    
    # Build all starting pairs: (N*k) chains, each starting with (i, knn[i,j])
    starts_i = torch.arange(N, device=device).unsqueeze(1).expand(-1, k).reshape(-1)  # (N*k,)
    starts_j = knn.reshape(-1)  # (N*k,)
    
    C = len(starts_i)  # total chains
    max_len = 50  # max chain length
    
    # Chain storage: (C, max_len) indices, -1 = empty
    chains = torch.full((C, max_len), -1, dtype=torch.long, device=device)
    chains[:, 0] = starts_i
    chains[:, 1] = starts_j
    chain_lens = torch.full((C,), 2, dtype=torch.long, device=device)
    
    # Used mask: (C, N) — which stars are used per chain
    used = torch.zeros(C, N, dtype=torch.bool, device=device)
    used[torch.arange(C, device=device), starts_i] = True
    used[torch.arange(C, device=device), starts_j] = True
    
    # Current direction per chain: (C, 3)
    direction = uv[starts_j] - uv[starts_i]
    dir_norm = torch.linalg.norm(direction, dim=1, keepdim=True).clamp(min=1e-12)
    direction = direction / dir_norm
    
    # Current last star per chain
    last = starts_j.clone()
    
    # Active mask
    active = torch.ones(C, dtype=torch.bool, device=device)
    # Deactivate chains where starts are the same point
    active &= (dir_norm.squeeze() > 1e-12)
    
    # Forward extension
    for step in range(max_len - 2):
        if not active.any():
            break
        
        n_active = active.sum().item()
        act_idx = torch.where(active)[0]  # indices of active chains
        
        # Vectors from last star to all stars: (n_active, N, 3)
        last_uv = uv[last[act_idx]]  # (n_active, 3)
        vecs = uv.unsqueeze(0) - last_uv.unsqueeze(1)  # (n_active, N, 3)
        norms = torch.linalg.norm(vecs, dim=2)  # (n_active, N)
        norms_safe = norms.clone()
        norms_safe[norms_safe < 1e-12] = float('inf')
        normed = vecs / norms_safe.unsqueeze(2)  # (n_active, N, 3)
        
        # Turning angles
        act_dir = direction[act_idx]  # (n_active, 3)
        cos_turns = (normed * act_dir.unsqueeze(1)).sum(dim=2)  # (n_active, N)
        
        # Valid: not used, valid turn angle
        act_used = used[act_idx]  # (n_active, N)
        valid = ~act_used & (cos_turns >= cos_max_turn) & (norms_safe < float('inf'))
        
        # Among valid, pick nearest
        big_dist = torch.full_like(norms, float('inf'))
        big_dist[valid] = norms[valid]
        nearest = big_dist.argmin(dim=1)  # (n_active,)
        has_valid = valid.any(dim=1)
        
        # Update active chains that found a next star
        found = act_idx[has_valid]
        if len(found) == 0:
            break
        
        next_stars = nearest[has_valid]
        cur_lens = chain_lens[found]
        
        # Store next star
        chains[found, cur_lens] = next_stars
        chain_lens[found] = cur_lens + 1
        used[found, next_stars] = True
        
        # Update direction
        new_dir = normed[has_valid, next_stars]
        direction[found] = new_dir
        last[found] = next_stars
        
        # Deactivate chains that didn't find a next star
        not_found = act_idx[~has_valid]
        active[not_found] = False
    
    # Now extend backward: reverse direction from start
    active_bwd = chain_lens >= 2  # all valid chains
    bwd_dir = uv[starts_i] - uv[starts_j]
    bwd_norm = torch.linalg.norm(bwd_dir, dim=1, keepdim=True).clamp(min=1e-12)
    bwd_dir = bwd_dir / bwd_norm
    bwd_last = starts_i.clone()
    
    # We'll prepend by storing backward extensions separately
    bwd_chains = torch.full((C, max_len), -1, dtype=torch.long, device=device)
    bwd_lens = torch.zeros(C, dtype=torch.long, device=device)
    
    for step in range(max_len - 2):
        if not active_bwd.any():
            break
        
        act_idx = torch.where(active_bwd)[0]
        n_active = len(act_idx)
        
        last_uv = uv[bwd_last[act_idx]]
        vecs = uv.unsqueeze(0) - last_uv.unsqueeze(1)
        norms = torch.linalg.norm(vecs, dim=2)
        norms_safe = norms.clone()
        norms_safe[norms_safe < 1e-12] = float('inf')
        normed = vecs / norms_safe.unsqueeze(2)
        
        act_dir = bwd_dir[act_idx]
        cos_turns = (normed * act_dir.unsqueeze(1)).sum(dim=2)
        
        act_used = used[act_idx]
        valid = ~act_used & (cos_turns >= cos_max_turn) & (norms_safe < float('inf'))
        
        big_dist = torch.full_like(norms, float('inf'))
        big_dist[valid] = norms[valid]
        nearest = big_dist.argmin(dim=1)
        has_valid = valid.any(dim=1)
        
        found = act_idx[has_valid]
        if len(found) == 0:
            break
        
        next_stars = nearest[has_valid]
        cur_bwd_lens = bwd_lens[found]
        
        bwd_chains[found, cur_bwd_lens] = next_stars
        bwd_lens[found] = cur_bwd_lens + 1
        used[found, next_stars] = True
        
        new_dir = normed[has_valid, next_stars]
        bwd_dir[found] = new_dir
        bwd_last[found] = next_stars
        
        not_found = act_idx[~has_valid]
        active_bwd[not_found] = False
    
    # Assemble full chains and deduplicate
    seen = set()
    result_chains = []
    
    chain_lens_cpu = chain_lens.cpu().numpy()
    bwd_lens_cpu = bwd_lens.cpu().numpy()
    chains_cpu = chains.cpu().numpy()
    bwd_chains_cpu = bwd_chains.cpu().numpy()
    
    for c in range(C):
        fwd_len = chain_lens_cpu[c]
        bwd_len = bwd_lens_cpu[c]
        total_len = fwd_len + bwd_len
        if total_len < min_stars:
            continue
        
        # Assemble: reversed backward + forward
        full = []
        for bi in range(bwd_len - 1, -1, -1):
            full.append(int(bwd_chains_cpu[c, bi]))
        for fi in range(fwd_len):
            full.append(int(chains_cpu[c, fi]))
        
        key = frozenset(full)
        if key not in seen:
            seen.add(key)
            result_chains.append(tuple(sorted(full)))
    
    return result_chains


SCORE_CUTOFFS = {'rms': 0.2, 'msd': 0.2}


def _batch_score_chains(chains_uv, chains_mag=None, max_mag=15.0):
    """Batch-score chains of the same length K on GPU.
    
    Args:
        chains_uv: (N, K, 3) tensor of unit vectors
        chains_mag: optional (N, K) tensor of magnitudes
        max_mag: limiting magnitude for normalizing mean_mag (from config)
    Returns:
        scores_dict: dict with 'rms', 'msd', 'smooth', 'smooth_mag' score tensors (N,)
        sort_orders: (N, K) tensor of indices sorting stars along the chain
    """
    N, K, _ = chains_uv.shape
    device = chains_uv.device

    # SVD for principal axis direction
    centroid = chains_uv.mean(dim=1, keepdim=True)
    centered = chains_uv - centroid
    _, _, Vh = torch.linalg.svd(centered, full_matrices=False)
    principal = Vh[:, 0, :]  # (N, 3)

    # Project onto principal axis and sort
    projections = (chains_uv * principal.unsqueeze(1)).sum(dim=2)  # (N, K)
    sort_order = torch.argsort(projections, dim=1)

    batch_idx = torch.arange(N, device=device).unsqueeze(1).expand(-1, K)
    sorted_uv = chains_uv[batch_idx, sort_order]

    # Straightness: RMS perpendicular distance from great circle through endpoints
    normal = torch.cross(sorted_uv[:, 0], sorted_uv[:, -1], dim=1)
    normal_norm = torch.linalg.norm(normal, dim=1, keepdim=True).clamp(min=1e-12)
    normal = normal / normal_norm

    interior = sorted_uv[:, 1:-1]  # (N, K-2, 3)
    dots = torch.abs((interior * normal.unsqueeze(1)).sum(dim=2)).clamp(0, 1)
    perp_deg = torch.rad2deg(torch.asin(dots))
    msd_perp = perp_deg.pow(2).mean(dim=1)
    rms_perp = msd_perp.sqrt()

    # Evenness: coefficient of variation of inter-star spacings
    cos_dots = (sorted_uv[:, :-1] * sorted_uv[:, 1:]).sum(dim=2).clamp(-1, 1)
    spacings = torch.acos(cos_dots)
    mean_sp = spacings.mean(dim=1)
    std_sp = spacings.std(dim=1)
    cv = torch.where(mean_sp > 1e-12, std_sp / mean_sp, torch.zeros_like(mean_sp))

    scores = {
        'rms': rms_perp + 0.3 * cv,
        'msd': msd_perp + 0.3 * cv,
    }

    # Smoothness: std of turning angles between consecutive segments
    dirs = sorted_uv[:, 1:] - sorted_uv[:, :-1]  # (N, K-1, 3)
    dir_norms = torch.linalg.norm(dirs, dim=2, keepdim=True).clamp(min=1e-12)
    dirs_normed = dirs / dir_norms
    cos_turn = (dirs_normed[:, :-1] * dirs_normed[:, 1:]).sum(dim=2).clamp(-1, 1)  # (N, K-2)
    turn_angles_deg = torch.rad2deg(torch.acos(cos_turn))
    turn_std = turn_angles_deg.std(dim=1)

    if chains_mag is not None:
        sorted_mag = chains_mag[batch_idx, sort_order]
        mean_mag = sorted_mag.mean(dim=1)
        std_mag = sorted_mag.std(dim=1)
        mag_cv = torch.where(mean_mag > 1e-12, std_mag / mean_mag, torch.zeros_like(mean_mag))
        scores['smooth'] = turn_std + 0.2 * cv + 0.3 * mag_cv
        # Magnitude gradient smoothness: std of consecutive magnitude differences
        mag_deltas = sorted_mag[:, 1:] - sorted_mag[:, :-1]  # (N, K-1)
        mag_gradient_std = mag_deltas.std(dim=1)
        scores['smooth_mag'] = turn_std + 0.2 * cv + 0.2 * mag_gradient_std
    else:
        scores['smooth'] = turn_std + 0.2 * cv
        scores['smooth_mag'] = turn_std + 0.2 * cv

    return scores, sort_order
def _chain_extent_batch(sorted_stars):
    """Vectorized angular extent for sorted chains (endpoint separation)."""
    ra0 = torch.deg2rad(sorted_stars[:, 0, 0])
    dec0 = torch.deg2rad(sorted_stars[:, 0, 1])
    ra1 = torch.deg2rad(sorted_stars[:, -1, 0])
    dec1 = torch.deg2rad(sorted_stars[:, -1, 1])
    cos_sep = (torch.sin(dec0) * torch.sin(dec1) +
               torch.cos(dec0) * torch.cos(dec1) * torch.cos(ra0 - ra1)).clamp(-1, 1)
    return torch.rad2deg(torch.acos(cos_sep))


def _score_chain(chain_uv):
    """Score a single chain's straightness + evenness. Returns (score, sorted_order)."""
    K = len(chain_uv)
    centroid = chain_uv.mean(dim=0)
    centered = chain_uv - centroid.unsqueeze(0)
    _, _, Vh = torch.linalg.svd(centered, full_matrices=False)
    projections = chain_uv @ Vh[0]
    sort_order = torch.argsort(projections)
    sorted_uv = chain_uv[sort_order]

    # Straightness
    normal = torch.cross(sorted_uv[0], sorted_uv[-1], dim=0)
    norm = torch.linalg.norm(normal)
    if norm < 1e-12:
        rms_perp = 0.0
    else:
        normal = normal / norm
        interior_idx = list(range(1, K - 1))
        if interior_idx:
            dots = torch.abs(sorted_uv[interior_idx] @ normal).clamp(0, 1)
            perp_deg = torch.rad2deg(torch.asin(dots))
            rms_perp = perp_deg.pow(2).mean().sqrt().item()
        else:
            rms_perp = 0.0

    # Evenness
    cos_dots = (sorted_uv[:-1] * sorted_uv[1:]).sum(dim=1).clamp(-1, 1)
    spacings = torch.acos(cos_dots)
    mean_sp = spacings.mean()
    cv = (spacings.std() / mean_sp).item() if mean_sp > 1e-12 else 0.0

    return rms_perp + 0.3 * cv, sort_order


def score_collinear_region(stars_tensor, device='cpu', mode='2d', angle_tol_deg=0.5):
    """Single-region collinear search (fallback for CPU path)."""
    stars_tensor = stars_tensor.to(device)
    N = len(stars_tensor)
    if N < 4:
        return []

    padded = stars_tensor.unsqueeze(0)  # (1, N, 3)
    counts = torch.tensor([N], dtype=torch.long)
    sa, si, unit_vecs = _batch_compute_angles(padded, counts, device)
    candidates = _find_chains_batch(sa, si, counts.numpy(), math.radians(angle_tol_deg))

    uv = unit_vecs[0]  # (N, 3)
    scored = []
    for _, chain_idx in candidates:
        idx_list = list(chain_idx)
        chain_uv = uv[idx_list]
        score, sort_order = _score_chain(chain_uv)
        chain_stars = stars_tensor[idx_list][sort_order]
        scored.append((score, chain_stars))

    scored.sort(key=lambda x: x[0])
    return scored[:5]



def process_collinear_regions(grid_points, gpu_catalog, config, device, max_extent=None):
    """Process ALL regions for collinear/smooth chains in batched GPU passes.
    
    Phase 1: Filter regions on GPU
    Phase 2a: Batch angle matrices on GPU + numpy chain finding (straight lines)
    Phase 2b: Greedy smooth chain finding (curves)
    Phase 3: Batch score all candidates on GPU, grouped by chain length
    """
    radius = config.search_radius_deg
    max_mag = config.max_mag
    max_stars = config.max_stars_per_region if config.max_stars_per_region > 0 else 200
    angle_tol_rad = math.radians(0.5)

    # --- Phase 1: Filter all regions, build padded tensor ---
    print("Phase 1: Filtering regions...")
    region_stars = []  # (grid_idx, stars_tensor)
    for idx, point in tqdm.tqdm(grid_points, desc="Filter"):
        ra, dec = point
        mask = (
            (gpu_catalog[:, 0] >= ra) & (gpu_catalog[:, 0] < ra + radius) &
            (gpu_catalog[:, 1] >= dec) & (gpu_catalog[:, 1] < dec + radius) &
            (gpu_catalog[:, 2] <= max_mag)
        )
        stars = gpu_catalog[mask]
        if len(stars) >= 4:
            if max_stars > 0 and len(stars) > max_stars:
                _, top_idx = torch.topk(stars[:, 2], max_stars, largest=False)
                stars = stars[top_idx]
            region_stars.append((idx, stars))

    if not region_stars:
        return pl.DataFrame()

    B = len(region_stars)
    actual_max_n = max(len(s) for _, s in region_stars)
    print(f"Phase 1 done: {B} regions with stars (max {actual_max_n}/region)")

    # --- Phase 2a: Batch angle computation on GPU + chain finding (straight lines) ---
    sub_batch_size = min(B, max(100, 4_000_000_000 // (actual_max_n * actual_max_n * 3 * 4)))
    print(f"Phase 2a: Computing angle matrices (sub-batch={sub_batch_size}, N={actual_max_n})...")

    all_candidates = []

    # Build concatenated star + unit_vec tensors for Phase 3 lookup
    concat_stars = torch.zeros(B, actual_max_n, 3, device=device)
    grid_indices = torch.zeros(B, dtype=torch.long, device=device)
    region_counts = torch.zeros(B, dtype=torch.long)
    for b, (idx, stars) in enumerate(region_stars):
        concat_stars[b, :len(stars)] = stars
        grid_indices[b] = idx
        region_counts[b] = len(stars)

    concat_uv_parts = []

    for sb_start in tqdm.tqdm(range(0, B, sub_batch_size), desc="Angles"):
        sb_end = min(sb_start + sub_batch_size, B)
        sb_B = sb_end - sb_start

        padded = concat_stars[sb_start:sb_end].clone()
        counts = region_counts[sb_start:sb_end].clone()

        sa, si, unit_vecs = _batch_compute_angles(padded, counts, device)
        candidates = _find_chains_batch(sa, si, counts.numpy(), angle_tol_rad)

        for b_local, chain_idx in candidates:
            global_b = sb_start + b_local
            all_candidates.append((global_b, chain_idx))

        concat_uv_parts.append(unit_vecs.cpu())
        del padded, sa, si, unit_vecs
        torch.cuda.empty_cache()

    # Concatenate unit vectors for Phase 3
    concat_uv = torch.cat(concat_uv_parts, dim=0).to(device)
    del concat_uv_parts

    n_straight = len(all_candidates)
    print(f"Phase 2a done: {n_straight:,} straight-line candidates")

    # --- Phase 2b: Greedy smooth chain finding (curves) ---
    print("Phase 2b: Finding smooth chains (curves)...")
    seen_global = set()
    for _, chain_idx in all_candidates:
        seen_global.add(chain_idx)

    n_smooth_new = 0
    for b in tqdm.tqdm(range(B), desc="Smooth"):
        n = int(region_counts[b])
        if n < 4:
            continue
        uv_np = concat_uv[b, :n].cpu().numpy()
        smooth_chains = _find_smooth_chains(uv_np, max_turn_deg=30.0, min_stars=4, k_neighbors=15)
        for chain_idx in smooth_chains:
            if chain_idx not in seen_global:
                seen_global.add(chain_idx)
                all_candidates.append((b, chain_idx))
                n_smooth_new += 1

    print(f"Phase 2b done: {n_smooth_new:,} new smooth candidates (total: {len(all_candidates):,})")

    if not all_candidates:
        return pl.DataFrame()

    # --- Phase 3: Batch scoring grouped by chain length ---
    print("Phase 3: Batch scoring on GPU...")
    by_length = defaultdict(list)
    for global_b, chain_idx in all_candidates:
        by_length[len(chain_idx)].append((global_b, list(chain_idx)))
    del all_candidates

    score_keys = ['rms', 'msd', 'smooth', 'smooth_mag']
    all_results = {k: [] for k in score_keys}
    all_results_regions = []
    all_results_lens = []
    all_results_stars = []

    for K in sorted(by_length.keys()):
        group = by_length[K]
        N_K = len(group)
        print(f"  {K}-chains: {N_K:,}")

        # Process in sub-batches to limit GPU memory (~500K per batch)
        score_batch_size = 500_000
        for g_start in range(0, N_K, score_batch_size):
            g_end = min(g_start + score_batch_size, N_K)
            batch = group[g_start:g_end]
            n_batch = len(batch)

            # Build index tensors for gathering
            gb = torch.tensor([g[0] for g in batch], device=device, dtype=torch.long)
            ci = torch.tensor([g[1] for g in batch], device=device, dtype=torch.long)

            # Gather unit vectors and star coords: (n_batch, K, 3)
            gb_exp = gb.unsqueeze(1).expand(-1, K)
            uv_batch = concat_uv[gb_exp, ci]
            stars_batch = concat_stars[gb_exp, ci]

            # Gather magnitudes: (n_batch, K)
            mag_batch = concat_stars[gb_exp, ci, 2]

            # Batch score on GPU
            scores_dict, sort_orders = _batch_score_chains(uv_batch, chains_mag=mag_batch, max_mag=max_mag)

            # Sort stars by position along chain
            bidx = torch.arange(n_batch, device=device).unsqueeze(1).expand(-1, K)
            sorted_stars = stars_batch[bidx, sort_orders]

            # Extent filter (vectorized, endpoint-based)
            if max_extent is not None:
                extents = _chain_extent_batch(sorted_stars)
                mask = extents <= max_extent
                for k in score_keys:
                    scores_dict[k] = scores_dict[k][mask]
                sorted_stars = sorted_stars[mask]
                gb = gb[mask]

            # Collect results (move to CPU)
            n_kept = len(scores_dict['rms'])
            if n_kept > 0:
                for k in score_keys:
                    all_results[k].append(scores_dict[k].cpu().numpy())
                all_results_regions.append(grid_indices[gb].cpu().numpy())
                all_results_lens.append(np.full(n_kept, K, dtype=np.int32))
                stars_cpu = sorted_stars.cpu().numpy()
                all_results_stars.extend(stars_cpu.tolist())

            del uv_batch, stars_batch, scores_dict, sort_orders, sorted_stars, gb, ci, mag_batch
            torch.cuda.empty_cache()

    del concat_uv, concat_stars, by_length

    if not all_results_regions:
        print("Phase 3 done: 0 chains after extent filter")
        return pl.DataFrame()

    regions_arr = np.concatenate(all_results_regions)
    lens_arr = np.concatenate(all_results_lens)

    print(f"Phase 3 done: {len(regions_arr):,} chains after extent filter")

    result = pl.DataFrame({
        "score_rms": np.concatenate(all_results['rms']).astype(np.float64),
        "score_msd": np.concatenate(all_results['msd']).astype(np.float64),
        "score_smooth": np.concatenate(all_results['smooth']).astype(np.float64),
        "score_smooth_mag": np.concatenate(all_results['smooth_mag']).astype(np.float64),
        "region": regions_arr.astype(np.int64),
        "chain_len": lens_arr.astype(np.int32),
        "stars": all_results_stars,
    })

    # Keep top 10 per region per chain_len, independently per scoring mode
    results = {}
    for k in score_keys:
        score_col = f"score_{k}"
        cutoff = SCORE_CUTOFFS.get(k, 1.0)
        df_k = result.select([
            pl.col(score_col).alias("score"),
            pl.col("region"), pl.col("chain_len"), pl.col("stars"),
        ]).filter(pl.col("score") <= cutoff)
        df_k = df_k.sort("score").group_by(["region", "chain_len"]).head(10)
        results[k] = df_k

    total = sum(len(v) for v in results.values())
    print(f"After top-10 filter + cutoffs: {total:,} chains (across {len(score_keys)} scoring modes)")

    return results

## Star filtering and grid utilities

In [7]:
#| export
def stars_for_point_and_radius(df, point, radius, max_mag):
    """ point is in the corner, not the center """
    ra, dec = point
    minra = ra
    maxra = ra + radius
    mindec = dec
    maxdec = dec + radius
    result = df.filter((pl.col("RAmdeg") < maxra) & (pl.col("RAmdeg") > minra) & (pl.col("DEmdeg") < maxdec) & (pl.col("DEmdeg") > mindec) & (pl.col("Vmag") <= max_mag))
    return result

def stars_for_center_and_radius(df, center, radius, max_mag):
    ra, dec = center
    minra = ra - radius/2
    maxra = ra + radius/2
    mindec = dec - radius/2
    maxdec = dec + radius/2
    return df.filter((pl.col("RAmdeg") < maxra) & (pl.col("RAmdeg") > minra) & (pl.col("DEmdeg") < maxdec) & (pl.col("DEmdeg") > mindec) & (pl.col("Vmag") <= max_mag))


def get_grid_points(config_or_step=None, min_dec=-90, max_dec=90):
    """Generate grid points for sky survey.
    
    Args:
        config_or_step: SearchConfig, int step size, or None (default 4deg)
    """
    if isinstance(config_or_step, SearchConfig):
        step = config_or_step.grid_step_deg
    elif isinstance(config_or_step, int):
        step = config_or_step
    elif config_or_step is None:
        step = 4
    else:
        step = int(config_or_step)
    RA_values = [ra for ra in range(0, 361, step)]
    Dec_values = [dec for dec in range(min_dec, max_dec+1, step)]
    grid_points = [(ra, dec) for ra in RA_values for dec in Dec_values]
    return grid_points

def get_grid_point_by_idx(idx, config=None):
    gp = get_grid_points(config)
    return gp[idx]

def get_region(df, idx, radius, max_mag, min_dec=-90, max_dec=90, config=None):
    center = get_grid_points(config, min_dec, max_dec)[idx]
    return stars_for_center_and_radius(df, center, radius, max_mag)

def get_center(dftycho, center, radius, max_mag):
    return stars_for_point_and_radius(dftycho, center, radius, max_mag)

def ra_to_hms(ra):
    if ra < 0.0:
        ra = ra + 360
    mm, hh = math.modf(ra / 15.0)
    _, mm = math.modf(mm * 60.0)
    ss = round(_ * 60.0)
    return hh, mm, ss

resultdf = pl.DataFrame({
    "score": pl.Float64,
    "region": pl.Int64,
    "item": []
})

## GPU pipeline

In [8]:
#| export

# Global GPU catalog cache
_gpu_catalog = None

def load_catalog_to_gpu(df, device='cuda'):
    """Load entire star catalog to GPU once for faster filtering"""
    global _gpu_catalog
    if _gpu_catalog is None and torch.cuda.is_available() and device == 'cuda':
        catalog_array = df.select(['RAmdeg', 'DEmdeg', 'Vmag']).to_numpy()
        _gpu_catalog = torch.tensor(catalog_array, dtype=torch.float32, device=device)
        print(f"Loaded {len(df):,} stars to GPU VRAM ({_gpu_catalog.element_size() * _gpu_catalog.nelement() / 1024**2:.1f} MB)")
    return _gpu_catalog

def reset_gpu_catalog():
    """Reset the GPU catalog cache (needed when switching catalogs or configs)."""
    global _gpu_catalog
    _gpu_catalog = None

def filter_stars_on_gpu(gpu_catalog, point, radius, max_mag):
    """Filter stars using GPU boolean masking - much faster than CPU Polars"""
    ra, dec = point
    minra, maxra = ra, ra + radius
    mindec, maxdec = dec, dec + radius
    
    mask = (
        (gpu_catalog[:, 0] < maxra) & 
        (gpu_catalog[:, 0] > minra) & 
        (gpu_catalog[:, 1] < maxdec) & 
        (gpu_catalog[:, 1] > mindec) & 
        (gpu_catalog[:, 2] <= max_mag)
    )
    return gpu_catalog[mask]

def vectorized_filter_batch(gpu_catalog, batch_points, config=None, radius=2, max_mag=15):
    """Filter ALL regions in batch simultaneously on GPU - NO Python loop"""
    if config is not None:
        radius = config.search_radius_deg
        max_mag = config.max_mag
    device = gpu_catalog.device
    
    # Convert batch points to tensor
    ra_starts = torch.tensor([p[0] for _, p in batch_points], device=device)
    dec_starts = torch.tensor([p[1] for _, p in batch_points], device=device)
    
    # Broadcast: catalog[N,3] vs batch[B] -> mask[B,N]
    # For each region, check if each star is in bounds
    ra_in_bounds = (gpu_catalog[:, 0].unsqueeze(0) >= ra_starts.unsqueeze(1)) & \
                   (gpu_catalog[:, 0].unsqueeze(0) < (ra_starts + radius).unsqueeze(1))
    dec_in_bounds = (gpu_catalog[:, 1].unsqueeze(0) >= dec_starts.unsqueeze(1)) & \
                    (gpu_catalog[:, 1].unsqueeze(0) < (dec_starts + radius).unsqueeze(1))
    mag_in_bounds = gpu_catalog[:, 2].unsqueeze(0) <= max_mag
    
    # Combined mask [B, N]
    mask = ra_in_bounds & dec_in_bounds & mag_in_bounds
    
    # Return list of star tensors per region
    result = []
    for i in range(len(batch_points)):
        region_stars = gpu_catalog[mask[i]]
        result.append((batch_points[i][0], region_stars))
    
    return result
        
def process(stars, region, point, nr_stars, device='cpu'):
    """Process on specified device - stars can be list or tensor"""
    if not isinstance(stars, torch.Tensor):
        stars = torch.tensor(stars, dtype=torch.float32, device=device)
    else:
        stars = stars.to(device)
    
    scores, points = mass_score_triangle_torch(stars, device=device)
    resultdf = pl.DataFrame({
        "score": scores.cpu().numpy(), 
        "region": [region] * len(scores),
        "points": points.cpu().numpy()
    })
    final_result_df = resultdf.top_k(5, by="score", reverse=True)
    print(f"Processed {region=} - {point} for {len(stars)} stars, {len(scores):,} triangles")
    return final_result_df

## Result saving

In [9]:
#| export
def add_to_result_and_save(resultdf, df: pl.DataFrame, filename):
    print("Saving result to", filename)
    has_tilt = "tilt" in df.columns
    for entry in df.iter_rows(named=True):
        stars_data = entry["points"] if isinstance(entry["points"], list) else entry["points"].tolist()
        row = {
            "score": entry["score"],
            "region": entry["region"],
            "stars": [stars_data],
        }
        if has_tilt:
            row["tilt"] = entry["tilt"]
        thisdf = pl.DataFrame(row)
        if resultdf.is_empty():
            resultdf = thisdf
        else:
            resultdf = pl.concat([resultdf, thisdf])
    resultdf.write_parquet(filename)
    return resultdf

schema = {
    "score": pl.Float64,
    "region": pl.Float64,
    "stars": pl.List(pl.List(pl.Float64)),
    "tilt": pl.Float64,
}

## Region processing pipeline

In [ ]:
#| export
def _score_region(stars_tensor, device, use_hip=False, score_triangles_hip=None,
                  score_squares_hip=None, mode='3d', shape='triangle'):
    """Score all combinations for a region, return top 5."""
    if shape == 'collinear':
        return score_collinear_region(stars_tensor, device=device, mode=mode)
    if shape == 'square':
        if use_hip and score_squares_hip is not None:
            scores, points_out = score_squares_hip(stars_tensor, mode=mode)
        else:
            scores, points_out = mass_score_square_torch(stars_tensor, device=device, mode=mode)
    elif use_hip and score_triangles_hip is not None:
        scores, points_out = score_triangles_hip(stars_tensor, mode=mode)
    else:
        scores, points_out = mass_score_triangle_torch(stars_tensor, device=device, mode=mode)
    k = min(5, len(scores))
    top5_scores, top5_idx = torch.topk(scores, k=k, largest=False)
    top5_points = points_out[top5_idx]
    return top5_scores, top5_points


def _cap_stars(stars_tensor, max_stars):
    """Keep only the brightest max_stars by sorting on magnitude (col 2)."""
    if max_stars <= 0 or len(stars_tensor) <= max_stars:
        return stars_tensor
    _, indices = torch.topk(stars_tensor[:, 2], max_stars, largest=False)
    return stars_tensor[indices]


def _detect_gpu_backend(hip_available=False):
    """Detect GPU and select best backend."""
    if not torch.cuda.is_available():
        return 'cpu', False
    gpu_name = torch.cuda.get_device_name(0).lower()
    is_amd = 'amd' in gpu_name or 'radeon' in gpu_name
    use_hip = hip_available and is_amd
    return 'cuda', use_hip


def process_all_regions(grid, df, config=None, start=0, end=0, batch_size=100, num_streams=4,
                        mode='3d', hip_available=False, score_triangles_hip=None,
                        score_squares_hip=None, shape='triangle'):
    """
    Process regions using the best available GPU backend.
    
    For collinear shape, delegates to process_collinear_regions() which batches
    all angle matrix computation into single GPU passes.
    """
    if end == 0:
        end = len(grid)

    radius = config.search_radius_deg if config else 2
    max_mag = config.max_mag if config else 15
    max_extent = config.max_extent_deg if config else None
    max_stars = config.max_stars_per_region if config else 0
    min_stars = 4

    zipped_list = list(zip(range(len(grid)), grid))
    grid_points = zipped_list[start:end]
    global_result = pl.DataFrame()

    device, use_hip = _detect_gpu_backend(hip_available)

    # --- Collinear: use dedicated batched pipeline ---
    if shape == 'collinear':
        if device != 'cuda':
            print("No GPU -- collinear requires CUDA")
            return global_result
        gpu_catalog = load_catalog_to_gpu(df, device)
        gpu_name = torch.cuda.get_device_name(0)
        config_name = config.name if config else "legacy"
        print(f"GPU: {gpu_name}")
        print(f"Config: {config_name} (mag<={max_mag}, radius={radius}deg, extent<={max_extent}deg, max_stars={max_stars})")
        print(f"Shape: collinear (batched GPU pipeline)")
        print(f"Regions: {len(grid_points)}")
        return process_collinear_regions(grid_points, gpu_catalog, config, device, max_extent)

    # --- Triangle/Square: existing stream-based pipeline ---
    if device != 'cuda':
        print("No GPU -- falling back to CPU (slow)")
        for idx, point in tqdm.tqdm(grid_points, desc="Regions"):
            stars = stars_for_point_and_radius(df, point=point, radius=radius, max_mag=max_mag)
            if len(stars) > 3:
                result = process(stars.rows(), idx, point, 3, device='cpu')
                if result is not None and not result.is_empty():
                    global_result = result if global_result.is_empty() else global_result.vstack(result)
        return global_result

    gpu_catalog = load_catalog_to_gpu(df, device)
    gpu_name = torch.cuda.get_device_name(0)

    # HIP kernels available for both triangles and squares
    hip_fn_available = (shape == 'triangle' and score_triangles_hip is not None) or \
                       (shape == 'square' and score_squares_hip is not None)
    actual_use_hip = use_hip and hip_fn_available

    if actual_use_hip:
        backend = "HIP kernel"
    else:
        backend = f"CUDA streams ({num_streams}x)"

    config_name = config.name if config else "legacy"
    print(f"GPU: {gpu_name}")
    print(f"Backend: {backend}")
    print(f"Config: {config_name} (mag<={max_mag}, radius={radius}deg, extent<={max_extent}deg, max_stars={max_stars})")
    print(f"Scoring: {mode}, shape: {shape}")
    print(f"Regions: {len(grid_points)}, batch size: {batch_size}")

    streams = [torch.cuda.Stream() for _ in range(num_streams)] if not actual_use_hip else []
    num_batches = (len(grid_points) + batch_size - 1) // batch_size

    for batch_idx in tqdm.tqdm(range(num_batches), desc="Batches"):
        batch_start = batch_idx * batch_size
        batch_end = min(batch_start + batch_size, len(grid_points))
        batch = grid_points[batch_start:batch_end]

        try:
            filtered_regions = vectorized_filter_batch(gpu_catalog, batch, config=config,
                                                       radius=radius, max_mag=max_mag)

            region_results = []

            if actual_use_hip:
                for idx, stars_tensor in filtered_regions:
                    stars_tensor = _cap_stars(stars_tensor, max_stars)
                    if len(stars_tensor) >= min_stars:
                        top5_s, top5_p = _score_region(stars_tensor, device, use_hip=True,
                                                       score_triangles_hip=score_triangles_hip,
                                                       score_squares_hip=score_squares_hip,
                                                       mode=mode, shape=shape)
                        region_results.append((idx, top5_s, top5_p))
                        del stars_tensor
            else:
                regions_per_stream = (len(filtered_regions) + num_streams - 1) // num_streams
                stream_results = [[] for _ in range(num_streams)]

                for stream_idx, stream in enumerate(streams):
                    s_start = stream_idx * regions_per_stream
                    s_end = min(s_start + regions_per_stream, len(filtered_regions))
                    if s_start >= len(filtered_regions):
                        break
                    with torch.cuda.stream(stream):
                        for i in range(s_start, s_end):
                            idx, stars_tensor = filtered_regions[i]
                            stars_tensor = _cap_stars(stars_tensor, max_stars)
                            if len(stars_tensor) >= min_stars:
                                top5_s, top5_p = _score_region(stars_tensor, device, use_hip=False,
                                                               mode=mode, shape=shape)
                                stream_results[stream_idx].append((idx, top5_s, top5_p))
                                del stars_tensor

                for stream in streams:
                    stream.synchronize()

                for sr in stream_results:
                    region_results.extend(sr)

            if len(region_results) == 0:
                torch.cuda.empty_cache()
                continue

            all_scores = []
            all_points = []
            all_regions = []
            for idx, scores, points in region_results:
                n = len(scores)
                all_scores.append(scores)
                all_points.append(points)
                all_regions.append(torch.full((n,), idx, dtype=torch.int32, device=device))

            scores_cat = torch.cat(all_scores, dim=0)
            points_cat = torch.cat(all_points, dim=0)
            regions_cat = torch.cat(all_regions, dim=0)
            del all_scores, all_points, all_regions, region_results

            if max_extent is not None and len(points_cat) > 0:
                extents = shape_extent_deg_batch(points_cat)
                extent_mask = extents <= max_extent
                scores_cat = scores_cat[extent_mask]
                points_cat = points_cat[extent_mask]
                regions_cat = regions_cat[extent_mask]

            if len(scores_cat) == 0:
                torch.cuda.empty_cache()
                continue

            tilt_cat = compute_tilt_batch(points_cat)

            batch_df = pl.DataFrame({
                "score": scores_cat.cpu().numpy(),
                "region": regions_cat.cpu().numpy(),
                "points": points_cat.cpu().numpy(),
                "tilt": tilt_cat.cpu().numpy(),
            })

            del scores_cat, points_cat, regions_cat
            torch.cuda.empty_cache()

            global_result = batch_df if global_result.is_empty() else pl.concat([global_result, batch_df])
            del batch_df

        except Exception as e:
            print(f"Batch {batch_idx + 1} error: {e}")
            torch.cuda.empty_cache()
            continue

    return global_result

In [11]:
#| hide
import nbdev; nbdev.nbdev_export()